In [1]:
import pandas as pd
import geopandas as gpd
from pathlib import Path
from parse_transit_lines import parse_transit_lines

#### extract transit links from the transit lines file

In [2]:
transit_links_df = parse_transit_lines()
display(transit_links_df.head())

e:\GitHub\TM1_Base_Network\utils\parse_transit_lines.py:70: UserWarning: LINE '21_5OUTA': missing ONEWAY; using F
  append_links()


,LINE NAME,MODE,ONEWAY,A,B,source
0,30_7AC,30,F,2412,2407,M:\Application\Model One\Networks\TM1_2015_Bas...
1,30_7AC,30,F,2407,2334,M:\Application\Model One\Networks\TM1_2015_Bas...
2,30_7AC,30,F,2334,2401,M:\Application\Model One\Networks\TM1_2015_Bas...
3,30_7AC,30,F,2401,12198,M:\Application\Model One\Networks\TM1_2015_Bas...
4,30_7AC,30,F,12198,11545,M:\Application\Model One\Networks\TM1_2015_Bas...


In [3]:
print('unique transit modes:', transit_links_df['MODE'].unique())

unique transit modes: [ 30  85  84  83  19  55 131 132 133 120 121  14  86  42  82  12  52 100
 101 102 103 104  87  70  16  20  21 110  93  95  60  68  24  66  81  28
  27  15  63  13  44  38  56  91  49  10  46  90  17  33 135 111  92 130
  18  88 107]


In [4]:
# local bus links
localbus_links_df = transit_links_df.loc[transit_links_df['MODE'] <= 78]
print('unique transit modes for local bus links:', localbus_links_df['MODE'].unique())
print('total local bus links, rows:', len(localbus_links_df))

# express bus links
expressbus_links_df = transit_links_df.loc[(transit_links_df['MODE'] >= 80) & (transit_links_df['MODE'] < 100)]
print('unique transit modes for express bus links:', expressbus_links_df['MODE'].unique())
print('total express bus links, rows:', len(expressbus_links_df))

unique transit modes for local bus links: [30 19 55 14 42 12 52 70 16 20 21 60 68 24 66 28 27 15 63 13 44 38 56 49
 10 46 17 33 18]
total local bus links, rows: 22052
unique transit modes for express bus links: [85 84 83 86 82 87 93 95 81 91 90 92 88]
total express bus links, rows: 5643


#### load roadway network links

In [5]:
roadway_links_df = gpd.read_file(
    Path.cwd().parent / "shapefiles" / "network_links.shp"
)
roadway_links_df = roadway_links_df[['A', 'B', 'FT', 'DISTANCE', 'SPDCLASS', 'CAPCLASS', 'LANES', 'USE']]
print('loaded roadway links, rows:', len(roadway_links_df))
print('unique A-B pairs:', roadway_links_df[['A', 'B']].drop_duplicates().shape[0])
display(roadway_links_df[['A', 'B', 'FT', 'DISTANCE', 'SPDCLASS', 'CAPCLASS', 'LANES', 'USE']].head())

c:\Users\ywang\AppData\Local\anaconda3\envs\rawfood\Lib\site-packages\pyogrio\core.py:34: RuntimeWarning: Could not detect GDAL data files. Set GDAL_DATA environment variable to the correct path.
  _init_gdal_data()


loaded roadway links, rows: 33953
unique A-B pairs: 33953


,A,B,FT,DISTANCE,SPDCLASS,CAPCLASS,LANES,USE
0,1,7487,6,0.05201,6,6,7,1
1,1,7489,6,0.04758,6,6,7,1
2,2,7496,6,0.06356,6,6,7,1
3,2,7500,6,0.05204,6,6,7,1
4,3,7483,6,0.12645,6,6,7,1


#### merge with local bus links

In [6]:
localbus_compare = localbus_links_df.merge(
    roadway_links_df,
    how='left',
    left_on=['A', 'B'],
    right_on=['A', 'B'],
    suffixes=('_transit', '_roadway'),
    indicator=True
)
# compare_links
print('total compared links, rows:', len(localbus_compare))
# print('unique A-B pairs in compared links:', localbus_compare[['A', 'B']].drop_duplicates().shape[0])

total compared links, rows: 22052


In [7]:
# unique A/B nodes associated with missing local bus links
print('number of missing local bus links:', localbus_compare.loc[localbus_compare['_merge'] == 'left_only'].shape[0])
print('unique A/B nodes associated with missing local bus links:',
      localbus_compare.loc[localbus_compare['_merge'] == 'left_only'][['A', 'B']].drop_duplicates().shape[0])

print('number of missing local bus links from two-way lines:', 
      localbus_compare.loc[(localbus_compare['_merge'] == 'left_only') & (localbus_compare['ONEWAY'] == 'F')].shape[0])
print('unique A/B nodes associated with missing local bus links from two-way lines:',
      localbus_compare.loc[(localbus_compare['_merge'] == 'left_only') & (localbus_compare['ONEWAY'] == 'F')][['A', 'B']].drop_duplicates().shape[0])

print('number of missing local bus links from one-way lines:', 
      localbus_compare.loc[(localbus_compare['_merge'] == 'left_only') & (localbus_compare['ONEWAY'] == 'T')].shape[0])
print('unique A/B nodes associated with missing local bus links from one-way lines:',
      localbus_compare.loc[(localbus_compare['_merge'] == 'left_only') & (localbus_compare['ONEWAY'] == 'T')][['A', 'B']].drop_duplicates().shape[0])

number of missing local bus links: 149
unique A/B nodes associated with missing local bus links: 130
number of missing local bus links from two-way lines: 147
unique A/B nodes associated with missing local bus links from two-way lines: 128
number of missing local bus links from one-way lines: 2
unique A/B nodes associated with missing local bus links from one-way lines: 2


In [8]:
display(localbus_compare.loc[localbus_compare['_merge'] == 'left_only'])

,LINE NAME,MODE,ONEWAY,A,B,source,FT,DISTANCE,SPDCLASS,CAPCLASS,LANES,USE,_merge
658,30_76AC,30,F,2412,2336,M:\Application\Model One\Networks\TM1_2015_Bas...,NaN,NaN,NaN,NaN,NaN,NaN,left_only
1992,30_805,30,F,3167,2938,M:\Application\Model One\Networks\TM1_2015_Bas...,NaN,NaN,NaN,NaN,NaN,NaN,left_only
2542,30_801AC,30,F,2818,2883,M:\Application\Model One\Networks\TM1_2015_Bas...,NaN,NaN,NaN,NaN,NaN,NaN,left_only
2543,30_801AC,30,F,2883,2884,M:\Application\Model One\Networks\TM1_2015_Bas...,NaN,NaN,NaN,NaN,NaN,NaN,left_only
2544,30_801AC,30,F,2884,2844,M:\Application\Model One\Networks\TM1_2015_Bas...,NaN,NaN,NaN,NaN,NaN,NaN,left_only
...,...,...,...,...,...,...,...,...,...,...,...,...,...
21364,24_278,24,F,6057,6059,E:\GitHub\NetworkWrangler\scripts\scratch\Wran...,NaN,NaN,NaN,NaN,NaN,NaN,left_only
21365,24_278,24,F,6059,6053,E:\GitHub\NetworkWrangler\scripts\scratch\Wran...,NaN,NaN,NaN,NaN,NaN,NaN,left_only
21366,24_278,24,F,6053,6054,E:\GitHub\NetworkWrangler\scripts\scratch\Wran...,NaN,NaN,NaN,NaN,NaN,NaN,left_only
21396,24_142,24,F,6504,6439,E:\GitHub\NetworkWrangler\scripts\scratch\Wran...,NaN,NaN,NaN,NaN,NaN,NaN,left_only


In [9]:
# source data

print('data sources for two-way bus lines')
print(localbus_compare.loc[(localbus_compare['_merge'] == 'left_only') & (localbus_compare['ONEWAY'] == 'F')]['source'].unique())

print('data sources for one-way bus lines')
print(localbus_compare.loc[(localbus_compare['_merge'] == 'left_only') & (localbus_compare['ONEWAY'] == 'T')]['source'].unique())

data sources for two-way bus lines
<ArrowStringArray>
[                              'M:\Application\Model One\Networks\TM1_2015_Base_Network\trn\transit_lines\AC_Local.tpl',
                            'M:\Application\Model One\Networks\TM1_2015_Base_Network\trn\transit_lines\CCCTA_local.tpl',
                              'M:\Application\Model One\Networks\TM1_2015_Base_Network\trn\transit_lines\GGT_Local.tpl',
                         'M:\Application\Model One\Networks\TM1_2015_Base_Network\trn\transit_lines\Muni_CableCars.tpl',
                             'M:\Application\Model One\Networks\TM1_2015_Base_Network\trn\transit_lines\MUNI_Local.tpl',
                            'M:\Application\Model One\Networks\TM1_2015_Base_Network\trn\transit_lines\SCVTA_Local.tpl',
      'E:\GitHub\NetworkWrangler\scripts\scratch\Wrangler_tmp_2026Mar31.132925\GTFS2023_SF_Muni\SF_2023_new_routes.lin',
       'E:\GitHub\NetworkWrangler\scripts\scratch\Wrangler_tmp_2026Mar31.132925\GTFS2023_SC_VTA\SC_

#### merge with express bus links

In [10]:
expressbus_compare = expressbus_links_df.merge(
    roadway_links_df,
    how='left',
    left_on=['A', 'B'],
    right_on=['A', 'B'],
    suffixes=('_transit', '_roadway'),
    indicator=True
)
# compare_links
print('total compared links, rows:', len(expressbus_compare))
# print('unique A-B pairs in compared links:', expressbus_compare[['A', 'B']].drop_duplicates().shape[0])

total compared links, rows: 5643


In [11]:
# unique A/B nodes associated with missing express bus links
print('number of missing express bus links:', expressbus_compare.loc[expressbus_compare['_merge'] == 'left_only'].shape[0])
print('unique A/B nodes associated with missing express bus links:',
      expressbus_compare.loc[expressbus_compare['_merge'] == 'left_only'][['A', 'B']].drop_duplicates().shape[0])

print('number of missing express bus links from two-way lines:', 
      expressbus_compare.loc[(expressbus_compare['_merge'] == 'left_only') & (expressbus_compare['ONEWAY'] == 'F')].shape[0])
print('unique A/B nodes associated with missing express bus links from two-way lines:',
      expressbus_compare.loc[(expressbus_compare['_merge'] == 'left_only') & (expressbus_compare['ONEWAY'] == 'F')][['A', 'B']].drop_duplicates().shape[0])

print('number of missing express bus links from one-way lines:', 
      expressbus_compare.loc[(expressbus_compare['_merge'] == 'left_only') & (expressbus_compare['ONEWAY'] == 'T')].shape[0])
print('unique A/B nodes associated with missing express bus links from one-way lines:',
      expressbus_compare.loc[(expressbus_compare['_merge'] == 'left_only') & (expressbus_compare['ONEWAY'] == 'T')][['A', 'B']].drop_duplicates().shape[0])

number of missing express bus links: 127
unique A/B nodes associated with missing express bus links: 78
number of missing express bus links from two-way lines: 93
unique A/B nodes associated with missing express bus links from two-way lines: 75
number of missing express bus links from one-way lines: 34
unique A/B nodes associated with missing express bus links from one-way lines: 3


In [12]:
display(expressbus_compare.loc[expressbus_compare['_merge'] == 'left_only'])

,LINE NAME,MODE,ONEWAY,A,B,source,FT,DISTANCE,SPDCLASS,CAPCLASS,LANES,USE,_merge
116,84_FWB,84,T,7016,8888,M:\Application\Model One\Networks\TM1_2015_Bas...,NaN,NaN,NaN,NaN,NaN,NaN,left_only
117,84_FEB,84,T,8888,7017,M:\Application\Model One\Networks\TM1_2015_Bas...,NaN,NaN,NaN,NaN,NaN,NaN,left_only
181,84_FSWB,83,T,7016,8888,M:\Application\Model One\Networks\TM1_2015_Bas...,NaN,NaN,NaN,NaN,NaN,NaN,left_only
182,84_FSEB,83,T,8888,7017,M:\Application\Model One\Networks\TM1_2015_Bas...,NaN,NaN,NaN,NaN,NaN,NaN,left_only
262,84_LWB,83,T,7016,8888,M:\Application\Model One\Networks\TM1_2015_Bas...,NaN,NaN,NaN,NaN,NaN,NaN,left_only
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1912,86_98XNB,86,F,2082,1973,M:\Application\Model One\Networks\TM1_2015_Bas...,NaN,NaN,NaN,NaN,NaN,NaN,left_only
1913,86_98XNB,86,F,1973,1974,M:\Application\Model One\Networks\TM1_2015_Bas...,NaN,NaN,NaN,NaN,NaN,NaN,left_only
1914,86_98XNB,86,F,1974,1954,M:\Application\Model One\Networks\TM1_2015_Bas...,NaN,NaN,NaN,NaN,NaN,NaN,left_only
4624,90_LynSB,90,T,7016,8888,M:\Application\Model One\Networks\TM1_2015_Bas...,NaN,NaN,NaN,NaN,NaN,NaN,left_only


In [13]:
# source data

print('data sources for two-way bus lines')
print(expressbus_compare.loc[(expressbus_compare['_merge'] == 'left_only') & (expressbus_compare['ONEWAY'] == 'F')]['source'].unique())

print('data sources for one-way bus lines')
print(expressbus_compare.loc[(expressbus_compare['_merge'] == 'left_only') & (expressbus_compare['ONEWAY'] == 'T')]['source'].unique())

data sources for two-way bus lines
<ArrowStringArray>
['M:\Application\Model One\Networks\TM1_2015_Base_Network\trn\transit_lines\CCCTA_EXP.tpl']
Length: 1, dtype: string
data sources for one-way bus lines
<ArrowStringArray>
['M:\Application\Model One\Networks\TM1_2015_Base_Network\trn\transit_lines\AC_TransBay.tpl', 'M:\Application\Model One\Networks\TM1_2015_Base_Network\trn\transit_lines\WESTCAT.tpl']
Length: 2, dtype: string
